# 端到端 Profiling：eager+packed 与 packed+varlen

本节比较当前版本里两条可运行的 Wordle SFT 路线。两路使用同一批 greedy-packed 数据、同一份 Qwen3-1.7B 权重、同一 seed 和同一批训练参数，唯一差别是 Attention 后端。

| 路线 | recipe | Attention | Packing |
|---|---|---|---|
| 基线 | `sft_qwen3_1_7b_wordle` | `flex`（`FlexAttention` + `BlockMask`，NPU 上 eager 执行） | greedy packing |
| 对比 | `sft_qwen3_1_7b_wordle_tnd` | `AscVarlenAttention` → CANN FA v3（TND 布局，`sparse_mode=7`） | 同一批 greedy-packed 数据 |

固定设置：2 卡（`dp_shard=2`，cp/tp/pp=1）、`local_batch_size=2`、`global_batch_size=4`、gradient accumulation 1、`seq_len=4096`、10 个 optimizer step，profiler 抓取第 5 步，两路都从 `assets/hf/Qwen3-1.7B` 载入基座权重。


In [ ]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/gitCode/cann/torchtitan-npu-wordle-latest
os.environ.update(dict(line.strip().split('=',1) for line in os.popen('source /home/developer/Ascend/cann/set_env.sh && env') if '=' in line))
os.environ["PATH"] = os.environ.get("PATH", "") + ":/usr/local/bin:/usr/local/sbin"
print('torchtitan root:', os.getcwd())


## 运行

两条命令只有 `--config` 和 varlen override 不同。注意 `--override.imports` 一旦在命令行给出，就会**替换** recipe 里的 override 列表；TND 这一路必须把 `torchtitan_npu.override.qwen3.varlen_attention.asc` 一并写出，否则会回落到上游 `VarlenAttention`（报 `aten::_flash_attention_forward` 不支持当前后端）。

```text
基线：--config sft_qwen3_1_7b_wordle
对比：--config sft_qwen3_1_7b_wordle_tnd
      --override.imports torchtitan_npu.override.qwen3.varlen_attention.asc
                          'torchtitan_npu.override.common.profiler.cann={"profile_ranks":[0],"profile_with_memory":true}'
```

`--metrics.log-freq 1` 让日志保留每个 step 的 loss / grad_norm / memory / tps，便于取稳态窗口。


In [ ]:
%%bash
set -euo pipefail
source /home/developer/Ascend/cann/set_env.sh
export PATH=/home/developer/.virtualenvs/python312/bin:$PATH

TTNPU_DIR="${TTNPU_DIR:-/mnt/workspace/gitCode/cann/torchtitan-npu-wordle-latest}"
HF_ASSETS_PATH="$TTNPU_DIR/assets/hf/Qwen3-1.7B"
PROFILER_OVERRIDE='torchtitan_npu.override.common.profiler.cann={"profile_ranks":[0],"profile_with_memory":true}'

run_route () {
  local config="$1" folder="$2" trace="$3"; shift 3
  rm -rf "$TTNPU_DIR/outputs/checkpoints/$folder" "$TTNPU_DIR/outputs/profile_traces/$trace"
  MODULE=torchtitan_npu.models.qwen3 CONFIG="$config" NGPU=2 \
  bash "$TTNPU_DIR/scripts/run_train.sh" \
    --hf-assets-path "$HF_ASSETS_PATH" \
    --checkpoint.folder "checkpoints/$folder" \
    --checkpoint.enable \
    --checkpoint.load-only \
    --checkpoint.initial-load-in-hf \
    --checkpoint.initial-load-path "$HF_ASSETS_PATH" \
    --training.steps 10 \
    --training.global-batch-size 4 \
    --training.seq-len 4096 \
    --metrics.log-freq 1 \
    --dataloader.dataset-path "assets/data/wordle" \
    --profiler.enable-profiling \
    --profiler.profile-freq 4 \
    --profiler.profiler-warmup 3 \
    --profiler.profiler-active 1 \
    --profiler.profiler-repeat 1 \
    --profiler.profiler-skip-first 1 \
    --profiler.save-traces-folder "profile_traces/$trace" \
    --override.imports "$@" "$PROFILER_OVERRIDE"
}

run_route sft_qwen3_1_7b_wordle ch6_eager_packed 06_eager_packed
run_route sft_qwen3_1_7b_wordle_tnd ch6_packed_varlen 06_packed_varlen \
  torchtitan_npu.override.qwen3.varlen_attention.asc


## 输出

Trace 保存到 `outputs/profile_traces/06_eager_packed` 与 `outputs/profile_traces/06_packed_varlen`，每个目录包含 `ASCEND_PROFILER_OUTPUT/operator_details.csv`（算子级 host / device 时长）、`trace_view.json`、`memory_record.csv`、`operator_memory.csv`。

profiler 窗口为 `profile_freq=4`、`warmup=3`、`active=1`、`repeat=1`、`skip_first=1`，因此两路抓取的都是第 5 步：这一步两路的输入相同、语义一致，可以直接比较 Attention 算子。


## 稳态结果（steps 7–10 均值）

首步包含编译，第 6 步包含 profiler 解析开销，两者都不计入；下表取 steps 7–10 的均值。

| 指标 | eager+packed（基线） | packed+varlen | 变化 |
|---|---:|---:|---|
| step time | 4.630 s | **1.411 s** | −69.5%（3.28×） |
| tps / device | 1,769.5 | **5,804.5** | +228.0% |
| memory（训练日志） | 33.43 GiB | **21.78 GiB** | −11.65 GiB（−34.8%） |
| MFU | 7.46% | 24.45% | +17.0 pt |
| loss | 0.7504 | 0.7503 | — |
| grad_norm | 5.3906 | 5.3939 | — |

逐步 loss 的最大差为 0.00246（step 3），step 10 差 0.00096：两路在同一条学习轨迹上，差别来自 Attention 后端的数值实现，而不是数据或超参。


## 第 5 步的 Attention 证据

| 路线 | Attention 算子（count / host total） | host 合计 |
|---|---|---:|
| eager+packed | `FlexAttentionAutogradOp` 56 / 1218.19 ms；`FlexAttentionAutogradOpBackward` 28 / 2315.79 ms | 3,533.98 ms |
| packed+varlen | `npu::npu_fusion_attention_v3` 112 / 27.90 ms；`npu::npu_fusion_attention_grad_v3` 28 / 5.17 ms；`aclnnFlashAttentionVarLenScore` 56 / 0.90 ms；`aclnnFlashAttentionUnpaddingScoreGrad` 28 / 0.30 ms | 34.27 ms |

基线由 `FlexAttentionAutogradOp` 承担 Attention，TND 这一路命中的是 NPU 融合算子与它对应的 ACLNN 下发（`aclnnFlashAttentionVarLenScore`、`aclnnFlashAttentionUnpaddingScoreGrad`）。同一个 profiled step 内，Attention 的 host 合计从 3,533.98 ms 降到 34.27 ms。

本环境 driver 25.5.5 采不到 device / AI-core 计数（`Device Duration` 列为 0），所以这里用同一 profiled step 内、口径一致的 host 合计做比较；算子名与调用次数来自 trace 本身。


## 历史记录（上一版本的三路对比）

上一版本的 06.06 比较过三条路线：`sft_qwen3_1_7b_wordle`（non-greedy causal SDPA）、`sft_qwen3_1_7b_wordle_block_causal_sdpa`（greedy block-causal SDPA）与 `sft_qwen3_1_7b_wordle_tnd`（greedy TND Varlen）。当时的结论是：block-causal mask 修复了 greedy packing 的语义，但仍走 dense SDPA，相对 causal 的 Attention device total 增加 70.2%、stage time 增加 7.1%；TND 相对 block-causal 把 Attention device total 降低 71.8%、stage time 降低 9.1%，训练日志 max reserved 少 1.74 GiB。

这两条 SDPA recipe（`sft_qwen3_1_7b_wordle_block_causal_sdpa` 与对应的 block-causal override）在当前版本已不存在，上述数字无法复现，只作为历史记录保留；本节的数据全部来自当前版本的两路运行。


In [ ]:
%cd $original_dir

## 本章小结

本章完成了从 Sequence Packing 到 TND Varlen Attention 的验证链路：先用 `positions` 表达 packed 样本的边界，再用 `cu_seqlens` 把同一批边界交给 TND kernel。

在当前版本的两路受控对比中，packed+varlen 相对 eager+packed 基线把稳态 step time 从 4.630 s 降到 1.411 s（3.28×），tps/device 从 1,769.5 提高到 5,804.5，训练日志 memory 从 33.43 GiB 降到 21.78 GiB（−34.8%），profiled step 内 Attention 算子 host 合计从 3,533.98 ms 降到 34.27 ms，两路 loss 的逐步最大差为 0.00246。

更早版本还比较过 non-greedy 与 greedy packing：greedy packing 把参考实验的 effective input / supervised token 吞吐提高约 2.38 倍、raw sample/s 提高约 2.25 倍。该对照的 recipe 在当前版本已不存在，只作历史参考。


## 练习

1. （单选题）06.06 对比的两条受控路线是什么？
    A. `sft_qwen3_1_7b_wordle`（eager+packed）与 `sft_qwen3_1_7b_wordle_tnd`（packed+varlen）
    B. DDP 与 TP
    C. eager 与量化
    D. FSDP 与 EP

2. （判断题）Profiler trace 负责证明实际算子路径；关闭 profiler 的重复 wall-time 更适合回答稳定的端到端速度。

3. （判断题）VarLen 跳过跨区间 Attention pair 后，padding 位置的 embedding、MLP 和 Norm 也会自动全部跳过。

4. （单选题）解释 packing 的端到端收益时，为什么不能只报告原生 TPS？
    A. 原生 TPS 只统计容器位置，可能掩盖 padding 与 supervised token 比例
    B. 原生 TPS 不包含任何 token
    C. 原生 TPS 只能在 CPU 上计算
    D. 原生 TPS 与 step time 无关


In [ ]:
!cat ./answer/06.06_answer.txt
